<a href="https://colab.research.google.com/github/vaishnavitv2007-cyber/1MS25IS136_assignment2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [27]:
import os

import os

key = None

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    key = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')
except Exception as e:
    print('Could not read from Colab Secrets:', type(e).__name__)

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
if not key:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')

if not isinstance(key, str) or not key.strip():
    raise ValueError('No API key found. Add GOOGLE_API_KEY in Colab Secrets or paste it when asked.')

os.environ['GOOGLE_API_KEY'] = key.strip()
print('API key set.')

API key loaded from Colab Secrets.
API key set.


In [28]:
# Quiet ensures your UI is not full of logs.
!pip install google-genai --quiet
print("Done.")

Done.


In [29]:
# @title User input needed.

DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic - sports, manga, food, music etc. "
        "Have a minimum of 4 sentences with each document. "
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "pasta_guide": (
        "Boil pasta in a large pot of well-salted water. "
        "Spaghetti takes about 9 to 11 minutes to cook until al dente. "
        "Save one cup of the starchy pasta water before draining. "
        "Toss the pasta with the sauce on medium heat for one minute so it coats evenly."
    ),
    "steak_guide": (
        "Take the steak out of the fridge 30 minutes before cooking. "
        "Heat a cast iron pan until it is very hot, around 230 degrees Celsius. "
        "Sear each side for 3 to 4 minutes for a medium-rare steak. "
        "Rest the steak for 5 minutes before slicing so the juices stay inside."
    ),
    "bread_guide": (
        "Basic bread dough needs flour, water, yeast and salt. "
        "Let the dough rise in a warm place for about one hour until it doubles in size. "
        "Bake the loaf in an oven preheated to 220 degrees Celsius. "
        "The bread is ready after 30 minutes when it sounds hollow if you tap the bottom."
    ),
    "rice_guide": (
        "Rinse basmati rice in cold water until the water runs clear. "
        "Use one cup of rice for every two cups of water. "
        "Bring the water to a boil, then cover the pot and simmer on low heat for 12 minutes. "
        "Turn off the heat and let the rice steam for 5 more minutes before fluffing with a fork."
    ),
}

In [30]:
# @title Define a tool
import re

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    words = set(re.findall(r"\w+", query.lower()))
    results = []

    for doc_name, content in DOCUMENTS.items():
        content_lower = content.lower()
        match_count = sum(1 for word in words if word in content_lower)
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for the search function.
print(search_documents("minimum of 4"))

[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic - sports, manga, food, music etc. Have a minimum of 4 sentences with each document. Duplicate this for a minimum of 4 documents to test your agent.

[Source: steak_guide]
Take the steak out of the fridge 30 minutes before cooking. Heat a cast iron pan until it is very hot, around 230 degrees Celsius. Sear each side for 3 to 4 minutes for a medium-rare steak. Rest the steak for 5 minutes before slicing so the juices stay inside.


In [31]:
# @title Ask gemini to search for something
from google import genai

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

# If you get a "404 model not found" error, uncomment the two lines below,
# run the cell, and copy a "flash" model name from the list into MODEL_NAME.
# for m in client.models.list():
#     print(m.name)
MODEL_NAME = "gemini-3.1-flash-lite"

SYSTEM_INSTRUCTION = '''
You are a chef training at a Michelin star restaurant.
When you need specific information about temperatures,
ingredients to use, cook time etc. use the search_documents tool.
Prefer answers based on what you find in the documents.
If the documents do not contain the answer, say so clearly,
and then answer from your general knowledge if you can.
'''

print("Model ready:", MODEL_NAME)

Model ready: gemini-3.1-flash-lite


In [32]:
import time
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.1-flash-lite"   # fast and working on your key

client = genai.Client(http_options=types.HttpOptions(timeout=60000))  # 60 seconds

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            print("  Sending request to the model...")
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config,
            )
        except Exception as e:
            msg = str(e)
            print(f"  Attempt {attempt + 1} failed: {msg[:200]}")
            retryable = any(x in msg for x in ["429", "503", "504", "timeout", "Timeout", "timed out"])
            if retryable and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Waiting {wait}s before retrying...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in (model_content.parts or []) if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
  Sending request to the model...
Step 1: model calls search_documents({'query': 'What topics are students supposed to share documents on?'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section. Documents can be related ...
  Sending request to the model...

Final answer:
According to the protocol, students are encouraged to share documents on any topic they choose—such as sports, manga, food, or music. Each document must be at least four sentences long, and students are required to provide a minimum of four documents in total.


In [33]:
# @title Ask follow up questions

questions = [
    "How long should I cook spaghetti and what should I save before draining?",
    "At what temperature should the oven be for baking bread, and how do I know it is done?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: How long should I cook spaghetti and what should I save before draining?
--------------------------------------------------
  Sending request to the model...
Step 1: model calls search_documents({'query': 'how long to cook spaghetti and what to save before draining'})
  Result preview: [Source: pasta_guide]
Boil pasta in a large pot of well-salted water. Spaghetti takes about 9 to 11 ...
  Sending request to the model...

Final answer:
To achieve the perfect texture, spaghetti should be boiled in well-salted water for **9 to 11 minutes** until it is *al dente*.

Before you drain the pasta, be sure to **save one cup of the starchy pasta water**. This liquid is essential for emulsifying your sauce and helping it coat the noodles evenly when you toss them together.

Question: At what temperature should the oven be for baking bread, and how do I know it is done?
--------------------------------------------------
  Sending request to the model...
Step 1: model calls search_docume

In [34]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)